In [1]:
from __future__ import annotations

import csv
import time
from copy import deepcopy
from pathlib import Path

import h5py
import numpy as np
import sympy as sp
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from config.ablation_config import DATA, ABLATION, CEQL_TRAIN, CEQL
from src.ComplexEQL import ComplexEQL
from src.utils import set_seed, train


class MSEOrRelativeMSELoss(nn.Module):
    def __init__(self, pivot: float = 1.0):
        super().__init__()
        self.pivot = pivot

    def forward(self, yhat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        residual = yhat - y
        denom = torch.maximum(y.abs(), y.new_tensor(self.pivot))
        return ((residual / denom) ** 2).mean()


def load_group(f: h5py.File, group_name: str):
    g = f[group_name]

    raw_expr = g["sympy_str"][()]
    true_expr_str = raw_expr.decode("utf-8") if isinstance(raw_expr, bytes) else str(raw_expr)

    X_train = g["train"]["X"][...].astype(np.float32, copy=False)
    y_train = g["train"]["y"][...].astype(np.float32, copy=False).reshape(-1)

    X_interp = g["test_interp"]["X"][...].astype(np.float32, copy=False)
    y_interp = g["test_interp"]["y"][...].astype(np.float32, copy=False).reshape(-1)

    X_extrap = g["test_extrap"]["X"][...].astype(np.float32, copy=False)
    y_extrap = g["test_extrap"]["y"][...].astype(np.float32, copy=False).reshape(-1)

    return true_expr_str, X_train, y_train, X_interp, y_interp, X_extrap, y_extrap


def mse(yhat: np.ndarray, y: np.ndarray) -> float:
    yhat = np.asarray(yhat, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    return float(np.mean((yhat - y) ** 2))


def snapshot_attrs(obj) -> dict:
    return deepcopy(obj.__dict__)


def restore_attrs(obj, state: dict) -> None:
    obj.__dict__.clear()
    obj.__dict__.update(deepcopy(state))


def add_two_logs_per_layer(no_params_list: list[list[dict]]) -> list[list[dict]]:
    new_list = deepcopy(no_params_list)

    for layer in new_list:
        n_logs = sum(node["op"] == "log" for node in layer)

        for _ in range(2 - n_logs):
            layer.insert(3, {"op": "log", "type": "unary"})

    return new_list


def configure_ablation(ablation_name: str) -> None:
    if ablation_name == "ceql_original":
        CEQL.use_skip_connections = True
        CEQL.weight_domain = "complex"

    elif ablation_name == "ceql_original_with_log":
        CEQL.use_skip_connections = True
        CEQL.weight_domain = "complex"
        CEQL.no_params_list = add_two_logs_per_layer(CEQL.no_params_list)
        CEQL.n_symbolic_layers = len(CEQL.no_params_list)

    elif ablation_name == "ceql_real_weights":
        CEQL.use_skip_connections = True
        CEQL.weight_domain = "real"

    elif ablation_name == "ceql_no_imaginary_penalty":
        CEQL.use_skip_connections = True
        CEQL.weight_domain = "complex"
        CEQL_TRAIN.imag_w_coeff_phase1 = 0.0
        CEQL_TRAIN.imag_w_coeff_phase2 = 0.0
        CEQL_TRAIN.imag_w_coeff_phase3 = 0.0

    elif ablation_name == "ceql_no_skip_connections":
        CEQL.use_skip_connections = False
        CEQL.weight_domain = "complex"

    else:
        raise ValueError(f"Unknown ablation: {ablation_name}")


def make_optimizer(model: torch.nn.Module) -> torch.optim.Optimizer:
    return torch.optim.Adam(
        model.parameters(),
        lr=CEQL_TRAIN.lr,
        betas=(0.9, 0.999),
    )


def make_scheduler(optimizer: torch.optim.Optimizer):
    if CEQL_TRAIN.scheduler == "ReduceLROnPlateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            **CEQL_TRAIN.schedulerparams,
        )

    return None


def make_dataloader(
    X_train: np.ndarray,
    y_train: np.ndarray,
    device: torch.device,
    seed: int,
) -> DataLoader:
    X_tensor = torch.tensor(X_train, device=device)
    y_tensor = torch.tensor(y_train.reshape(-1, 1), device=device)

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        TensorDataset(X_tensor, y_tensor),
        batch_size=CEQL_TRAIN.train_batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=0,
        generator=generator,
    )


def symbol_list(n_features: int) -> list[sp.Symbol]:
    return [sp.Symbol(f"x{i + 1}") for i in range(n_features)]


def symbol_dict(n_features: int) -> dict[str, sp.Symbol]:
    symbols = symbol_list(n_features)
    return {str(symbol): symbol for symbol in symbols}


def sympy_count_ops(expr_str: str, n_features: int) -> int:
    if expr_str == "":
        return -1

    expr = sp.sympify(expr_str, locals=symbol_dict(n_features))
    return int(sp.count_ops(expr, visual=False))


def symbolic_match(true_expr_str: str, found_expr_str: str, n_features: int) -> bool:
    if found_expr_str == "":
        return False

    variables = symbol_dict(n_features)

    true_expr = sp.nsimplify(sp.sympify(true_expr_str, locals=variables))
    found_expr = sp.nsimplify(sp.sympify(found_expr_str, locals=variables))

    diff = true_expr - found_expr

    checks = [
        sp.simplify(diff),
        sp.expand(diff),
        sp.factor(diff),
        sp.cancel(sp.together(diff)),
    ]

    return any(check == 0 for check in checks)


def predict_numpy(model: torch.nn.Module, X: np.ndarray, device: torch.device) -> np.ndarray:
    X_tensor = torch.tensor(X, device=device)

    with torch.no_grad():
        yhat = model(X_tensor).real.squeeze(-1).cpu().numpy()

    return yhat


def library_string(no_params_list: list[list[dict]]) -> str:
    return " | ".join(
        ",".join(node["op"] for node in layer)
        for layer in no_params_list
    )


def main() -> None:
    out_csv = Path(ABLATION.results_path)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    device = torch.device(CEQL_TRAIN.device)

    ceql_state0 = snapshot_attrs(CEQL)
    train_state0 = snapshot_attrs(CEQL_TRAIN)

    with h5py.File(DATA.h5_path, "r") as f, out_csv.open("w", newline="") as out:
        writer = csv.writer(out)

        writer.writerow(
            [
                "ablation",
                "group",
                "run",
                "seed",
                "n_train",
                "n_features",
                "use_skip_connections",
                "weight_domain",
                "library",
                "imag_w_coeff_phase1",
                "imag_w_coeff_phase2",
                "imag_w_coeff_phase3",
                "train_mse",
                "test_interp_mse",
                "test_extrap_mse",
                "duration_s",
                "symbolic_match",
                "sympy_count_ops",
                "true_expr",
                "found_expr",
            ]
        )

        all_groups = sorted(list(f.keys()))
        selected_groups = [all_groups[i] for i in ABLATION.benchmark_group_indices]

        print("All benchmark groups:")
        print(all_groups)
        print()
        print("Selected benchmark groups:")
        print(selected_groups)

        for group_name in selected_groups:
            print()
            print("#" * 80)
            print(f"Running benchmark group: {group_name}")
            print("#" * 80)

            (
                true_expr_str,
                X_train,
                y_train,
                X_interp,
                y_interp,
                X_extrap,
                y_extrap,
            ) = load_group(f, group_name)

            n_features = int(X_train.shape[1])

            for ablation_name in ABLATION.ablations:
                restore_attrs(CEQL, ceql_state0)
                restore_attrs(CEQL_TRAIN, train_state0)

                CEQL.n_input_fields = n_features
                configure_ablation(ablation_name)

                library = library_string(CEQL.no_params_list)

                print()
                print("=" * 80)
                print(f"Ablation: {ablation_name}")
                print(f"Group: {group_name}")
                print(f"use_skip_connections: {CEQL.use_skip_connections}")
                print(f"weight_domain: {CEQL.weight_domain}")
                print(f"library: {library}")
                print(f"imag_w_coeff_phase1: {CEQL_TRAIN.imag_w_coeff_phase1}")
                print(f"imag_w_coeff_phase2: {CEQL_TRAIN.imag_w_coeff_phase2}")
                print(f"imag_w_coeff_phase3: {CEQL_TRAIN.imag_w_coeff_phase3}")
                print("=" * 80)

                for run_idx in range(ABLATION.n_runs):
                    seed = ABLATION.base_seed + run_idx
                    set_seed(seed)

                    dataloader = make_dataloader(X_train, y_train, device, seed)

                    model = ComplexEQL(CEQL).to(device)
                    loss_fn = MSEOrRelativeMSELoss(pivot=1.0)

                    optimizer = make_optimizer(model)
                    scheduler = make_scheduler(optimizer)

                    start = time.perf_counter()

                    model, _ = train(
                        model=model,
                        dataloader=dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        cfg=CEQL_TRAIN,
                        device=device,
                        scheduler=scheduler,
                        on_print=None,
                    )

                    duration = time.perf_counter() - start

                    model.eval()

                    yhat_train = predict_numpy(model, X_train, device)
                    yhat_interp = predict_numpy(model, X_interp, device)
                    yhat_extrap = predict_numpy(model, X_extrap, device)

                    train_mse = mse(yhat_train, y_train)
                    test_interp_mse = mse(yhat_interp, y_interp)
                    test_extrap_mse = mse(yhat_extrap, y_extrap)

                    symbols = symbol_list(n_features)
                    found_expr = model.get_symbolic_expression(
                        symbols,
                        rounding_decimals=5,
                        use_imag=False,
                    )

                    found_expr_str = "" if found_expr is None else str(found_expr)
                    match = symbolic_match(true_expr_str, found_expr_str, n_features)
                    count_ops = sympy_count_ops(found_expr_str, n_features)

                    writer.writerow(
                        [
                            ablation_name,
                            group_name,
                            run_idx,
                            seed,
                            int(X_train.shape[0]),
                            n_features,
                            CEQL.use_skip_connections,
                            CEQL.weight_domain,
                            library,
                            CEQL_TRAIN.imag_w_coeff_phase1,
                            CEQL_TRAIN.imag_w_coeff_phase2,
                            CEQL_TRAIN.imag_w_coeff_phase3,
                            train_mse,
                            test_interp_mse,
                            test_extrap_mse,
                            duration,
                            int(match),
                            count_ops,
                            true_expr_str,
                            found_expr_str,
                        ]
                    )

                    out.flush()

                    print(
                        f"[{ablation_name} | {group_name}] "
                        f"run={run_idx} seed={seed} "
                        f"train={train_mse:.3e} "
                        f"interp={test_interp_mse:.3e} "
                        f"extrap={test_extrap_mse:.3e} "
                        f"match={int(match)} "
                        f"ops={count_ops} "
                        f"dur={duration:.2f}s"
                    )

                    print(f"Found: {found_expr_str}")

    restore_attrs(CEQL, ceql_state0)
    restore_attrs(CEQL_TRAIN, train_state0)

    print()
    print(f"Saved: {out_csv}")


if __name__ == "__main__":
    main()

All benchmark groups:
['A1_three_variate_linear_rational']

Selected benchmark groups:
['A1_three_variate_linear_rational']

################################################################################
Running benchmark group: A1_three_variate_linear_rational
################################################################################

Ablation: ceql_original
Group: A1_three_variate_linear_rational
use_skip_connections: True
weight_domain: complex
library: id,id,id,const,const,square,square,div,div | id,id,id,const,const,square,square,div,div
imag_w_coeff_phase1: 1e-10
imag_w_coeff_phase2: 1e-05
imag_w_coeff_phase3: 0.01
Random seed set as 0
[PHASE1 | Epoch 1] lr=1.00e-03, total=1.1508e+02, data=1.1508e+02, sparsity_reg=4.6087e-09, imag_w=1.1832e-11, theta=0.0000e+00, valid=1.000, active_edges=177
[PHASE1 | Epoch 1000] lr=1.00e-03, total=3.0707e-01, data=3.0707e-01, sparsity_reg=4.3402e-09, imag_w=1.4527e-11, theta=0.0000e+00, valid=1.000, active_edges=177
[PHASE1 | Epoch 2000]